# 在 TRL 中实现 GRPO

本节将学习如何使用 **TRL（Transformer Reinforcement Learning）** 库实现 GRPO 训练。TRL 封装了 GRPO 的所有复杂性，让我们只需关注业务逻辑（数据集和奖励函数）。

> **TIP**: 本节面向 TRL 初学者。如果你已经熟悉 TRL，可以直接查看 [Open R1 的 GRPO 实现](https://github.com/huggingface/open-r1/blob/main/src/open_r1/grpo.py)。

使用 TRL 实现 GRPO 只需四步：
1. 准备 **数据集**（prompts）
2. 定义 **奖励函数**（评估生成质量）
3. 配置 **GRPOConfig**（训练超参数）
4. 用 **GRPOTrainer** 训练

In [ ]:
# 安装依赖（在 Colab 或首次使用时运行）
# !pip install trl datasets transformers accelerate

## 最小化 GRPO 示例

以下是一个完整的、可运行的最简 GRPO 训练示例：

In [ ]:
from trl import GRPOTrainer, GRPOConfig
from datasets import load_dataset

# --------------------------------------------------------
# 步骤 1：加载数据集
# 数据集应包含 prompt 字段，模型将对每个 prompt 生成多个回答
# --------------------------------------------------------
# dataset = load_dataset("your_dataset", split="train")

# --------------------------------------------------------
# 步骤 2：定义奖励函数
# 奖励函数接收 completions 列表，返回对应的 rewards 列表
# 这里用「回答长度」作为简单的奖励信号
# --------------------------------------------------------
def reward_func(completions, **kwargs):
    """
    简单的长度奖励函数示例：越长的回答奖励越高
    
    参数：
        completions: 模型生成的候选答案列表
        **kwargs: 其他可能的参数（如 prompts、answers 等）
    
    返回：
        List[float]: 每个候选答案对应的奖励值
    """
    # 以字符长度作为奖励值（实际任务中应使用更有意义的指标）
    return [float(len(completion)) for completion in completions]

# --------------------------------------------------------
# 步骤 3：配置训练参数
# --------------------------------------------------------
training_args = GRPOConfig(
    output_dir="output",           # 模型和日志的输出目录
    num_train_epochs=3,            # 训练轮数
    per_device_train_batch_size=4, # 每个设备的 batch size（应能容纳所有生成结果）
    gradient_accumulation_steps=2, # 梯度累积步数（等效于更大的 batch size）
    logging_steps=10,              # 每隔多少步记录一次日志
)

# --------------------------------------------------------
# 步骤 4：初始化并启动训练
# --------------------------------------------------------
# trainer = GRPOTrainer(
#     model="your_model",           # 模型名称或本地路径，如 "Qwen/Qwen2-0.5B-Instruct"
#     args=training_args,
#     train_dataset=dataset,
#     reward_funcs=reward_func,     # 奖励函数（可传入列表使用多个）
# )
# trainer.train()

print("GRPO 最小化示例代码结构演示完成")
print("实际运行需要提供真实的模型名称和数据集")

## 奖励函数设计指南

奖励函数是 GRPO 训练的核心，决定了模型朝什么方向优化。DeepSeek R1 论文中展示了三种有效的奖励函数设计思路：

### 类型一：基于长度的奖励

最简单的奖励函数，控制模型输出的长度：

In [1]:
def reward_len(completions, **kwargs):
    """
    长度控制奖励函数：鼓励模型生成接近目标长度的回答
    
    奖励值为负，绝对值越小越好（越接近目标长度）：
    - 奖励 = -|ideal_length - len(completion)|
    - 完全匹配目标长度 → 奖励 = 0（最高）
    - 偏差越大 → 奖励越负（越低）
    """
    ideal_length = 20  # 目标回答长度（字符数）
    return [-abs(ideal_length - len(completion)) for completion in completions]


# 测试奖励函数
test_completions = [
    "短",                          # 长度 1
    "这是一个恰好二十字符长的回答！",   # 长度约 20
    "这是一个非常非常非常非常非常非常长的回答，远超目标长度了",  # 长度 > 20
]

rewards = reward_len(test_completions)
print("长度控制奖励函数测试：")
for completion, reward in zip(test_completions, rewards):
    print(f"  长度={len(completion):3d}, 奖励={reward:5.1f}, 文本：'{completion[:20]}...' ")

长度控制奖励函数测试：
  长度=  1, 奖励=-19.0, 文本：'短...' 
  长度= 15, 奖励= -5.0, 文本：'这是一个恰好二十字符长的回答！...' 
  长度= 28, 奖励= -8.0, 文本：'这是一个非常非常非常非常非常非常长的回答...' 


### 类型二：基于规则的奖励（可验证任务）

对于答案可客观验证的任务（数学、编程等），使用规则函数给予准确的奖励：

In [2]:
def extract_final_answer(completion: str) -> str:
    """
    从模型输出中提取最终答案
    实际使用时需要根据模型输出格式定制解析逻辑
    """
    # 简化示例：假设答案在最后一行
    lines = completion.strip().split('\n')
    return lines[-1].strip()


def problem_reward(completions, answers, **kwargs):
    """
    数学题验证奖励函数：答案正确给予奖励
    
    参数：
        completions: 模型生成的完整回答列表
        answers: 数据集中对应的正确答案列表
        **kwargs: 其他参数
    
    返回：
        List[float]: 二元奖励（1.0=正确，0.0=错误）
    """
    rewards = []
    for completion, correct_answer in zip(completions, answers):
        try:
            # 从模型输出中提取最终答案
            answer = extract_final_answer(completion)
            
            # 二元奖励：正确得 1，错误得 0
            # 实际使用时可以添加部分奖励（如步骤正确但答案错）
            reward = 1.0 if answer == correct_answer else 0.0
            rewards.append(reward)
        except Exception:
            # 无法解析答案时给予最低奖励
            rewards.append(0.0)
    
    return rewards


# 测试
test_completions = [
    "先计算 2×6=12，再加 2，所以答案是\n14",   # 正确
    "直接加减：2+2=4，4×6=24，所以是\n24",     # 错误
    "没有解题过程\n14",                         # 正确（答案对但过程省略）
]
correct_answers = ["14", "14", "14"]

rewards = problem_reward(test_completions, correct_answers)
print("数学题验证奖励函数测试：")
for i, (completion, reward) in enumerate(zip(test_completions, rewards)):
    status = "✓" if reward == 1.0 else "✗"
    print(f"  候选 {i+1}: 奖励={reward} {status}")
    print(f"    回答摘要：'{completion.split(chr(10))[-1].strip()}'")

数学题验证奖励函数测试：
  候选 1: 奖励=1.0 ✓
    回答摘要：'14'
  候选 2: 奖励=0.0 ✗
    回答摘要：'24'
  候选 3: 奖励=1.0 ✓
    回答摘要：'14'


### 类型三：基于格式的奖励

DeepSeek R1 训练中的关键奖励之一：鼓励模型遵循「先思考再回答」的结构化格式：

In [3]:
import re

def format_reward(completions, **kwargs):
    """
    格式奖励函数：鼓励模型使用 <think>...</think><answer>...</answer> 格式
    
    奖励规则：
    - 完全符合格式 + 内容充实 → 1.0（满分）
    - 符合格式但内容不足 → 0.5（部分奖励）
    - 不符合格式 → 0.0（无奖励）
    """
    # 使用正则表达式检查是否包含 <think> 和 <answer> 标签
    pattern = r"<think>(.*?)</think>\s*<answer>(.*?)</answer>"
    
    rewards = []
    for completion in completions:
        match = re.search(pattern, completion, re.DOTALL)  # re.DOTALL 使 . 匹配换行符
        if match:
            think_content = match.group(1).strip()   # 提取思考内容
            answer_content = match.group(2).strip()  # 提取答案内容
            
            # 思考内容 >20 字符且答案非空 → 满分
            if len(think_content) > 20 and len(answer_content) > 0:
                rewards.append(1.0)
            else:
                # 格式正确但内容不充实 → 部分奖励
                rewards.append(0.5)
        else:
            # 格式不正确 → 无奖励
            rewards.append(0.0)
    
    return rewards


# 测试不同格式的回答
test_completions = [
    # 格式正确，内容充实
    "<think>\n首先按运算优先级，先算乘法 2×6=12，再加 2，得 14\n</think>\n<answer>14</answer>",
    
    # 格式正确，但思考内容太短
    "<think>算一下</think><answer>14</answer>",
    
    # 格式不正确（无标签）
    "答案是 14",
    
    # 格式不正确（只有部分标签）
    "<think>计算中...</think> 结果是 14",
]

rewards = format_reward(test_completions)
print("格式奖励函数测试：")
for i, (completion, reward) in enumerate(zip(test_completions, rewards)):
    print(f"  候选 {i+1}: 奖励={reward}")
    print(f"    内容摘要：'{completion[:50]}...'")
    print()

格式奖励函数测试：
  候选 1: 奖励=1.0
    内容摘要：'<think>
首先按运算优先级，先算乘法 2×6=12，再加 2，得 14
</think>
<a...'

  候选 2: 奖励=0.5
    内容摘要：'<think>算一下</think><answer>14</answer>...'

  候选 3: 奖励=0.0
    内容摘要：'答案是 14...'

  候选 4: 奖励=0.0
    内容摘要：'<think>计算中...</think> 结果是 14...'



## GRPOConfig 关键参数详解

以下是最重要的训练配置参数及其说明：

In [ ]:
from trl import GRPOConfig

# GRPOConfig 完整配置示例（带详细注释）
training_args = GRPOConfig(
    # ---- 基础训练参数 ----
    output_dir="output",                 # 模型检查点和日志保存目录
    num_train_epochs=3,                  # 训练总轮数
    per_device_train_batch_size=4,       # 每个 GPU 的 batch size
                                         # 注意：需要能容纳 num_generations 个生成结果
    gradient_accumulation_steps=2,       # 梯度累积步数（等效 batch size = 4×2=8）
    learning_rate=1e-5,                  # 学习率（GRPO 通常比 SFT 更小）
    logging_steps=10,                    # 每隔多少步打印日志
    
    # ---- GRPO 核心参数 ----
    num_generations=4,                   # 每个 prompt 生成的候选数（组大小 G）
                                         # 太小（<4）：多样性不足，梯度不稳定
                                         # 推荐（4~16）：在多样性和计算成本间平衡
                                         # 太大（>16）：计算成本急剧增加
    max_prompt_length=512,               # prompt 的最大 token 长度
    max_completion_length=256,           # 生成回答的最大 token 长度
    
    # ---- 加速选项 ----
    use_vllm=False,                      # 是否使用 vLLM 加速推理（支持的模型效果显著）
    bf16=True,                           # 使用 bfloat16 精度（现代 GPU 推荐）
)

print("GRPOConfig 关键参数说明：")
print()
print("num_generations（组大小 G）的选择建议：")
print("  2~3：多样性不足，通常不够好")
print("  4~8：适合简单任务或资源受限场景")
print(" 8~16：适合复杂推理任务（如数学）")
print("  >16：计算成本过高，收益递减")
print()
print("训练监控指标含义：")
print("  reward      : 奖励函数的平均奖励值（应随训练上升）")
print("  reward_std  : 奖励的组内标准差（反映生成多样性）")
print("  kl          : 与参考模型的 KL 散度（过大说明偏离过多）")
print("  loss        : GRPO 损失（随训练增大是正常现象，见 Section 05 解释）")

## 多奖励函数联合使用

GRPOTrainer 支持同时使用多个奖励函数，最终奖励是各函数输出的**加和**：

In [ ]:
# 联合使用多个奖励函数的示例
# GRPOTrainer 会将各奖励函数的结果加总作为最终奖励

from trl import GRPOTrainer, GRPOConfig

# 奖励函数 1：格式奖励（最高 1.0 分）
def reward_format(completions, **kwargs):
    pattern = r"<think>.*?</think>\s*<answer>.*?</answer>"
    return [1.0 if re.match(pattern, c, re.DOTALL) else 0.0 for c in completions]

# 奖励函数 2：正确性奖励（最高 2.0 分，权重更高）
def reward_correctness(completions, answers, **kwargs):
    rewards = []
    for completion, correct in zip(completions, answers):
        # 简单判断：提取 <answer> 标签中的内容
        match = re.search(r"<answer>(.*?)</answer>", completion, re.DOTALL)
        if match and match.group(1).strip() == correct:
            rewards.append(2.0)  # 正确答案，高权重奖励
        else:
            rewards.append(0.0)
    return rewards

# 初始化 Trainer 时传入奖励函数列表
# trainer = GRPOTrainer(
#     model="Qwen/Qwen2-0.5B-Instruct",
#     args=training_args,
#     train_dataset=dataset,
#     reward_funcs=[reward_format, reward_correctness],  # 多个奖励函数
# )

# 测试联合奖励
test_completions = [
    "<think>先算乘法再加法</think><answer>14</answer>",  # 格式+正确：1.0+2.0=3.0
    "<think>直接加起来</think><answer>16</answer>",      # 格式+错误：1.0+0.0=1.0  
    "答案就是 14",                                       # 无格式+正确：0.0+0.0=0.0
]
correct_answers = ["14", "14", "14"]

format_rewards = reward_format(test_completions)
correctness_rewards = reward_correctness(test_completions, correct_answers)
total_rewards = [f + c for f, c in zip(format_rewards, correctness_rewards)]

print("联合奖励函数测试：")
print(f"{'候选':^4} {'格式奖励':^8} {'正确性奖励':^10} {'总奖励':^8}")
print("-" * 34)
for i, (f, c, t) in enumerate(zip(format_rewards, correctness_rewards, total_rewards)):
    print(f"  {i+1}      {f:^8.1f}    {c:^10.1f}    {t:^8.1f}")

## 训练技巧与注意事项

### 显存管理

GRPO 需要同时存储多个生成结果，显存需求比普通微调更高：

| 内存优化策略 | 方法 |
|------------|------|
| 减小 batch size | 降低 `per_device_train_batch_size` |
| 减少生成数量 | 降低 `num_generations`（最低 4）|
| 使用梯度累积 | 增大 `gradient_accumulation_steps` |
| 使用 LoRA | 减少可训练参数数量（见 Section 05）|
| 使用 4-bit 量化 | `load_in_4bit=True`（见 Section 06）|

### 训练稳定性

> **WARNING**: GRPO 训练中，**loss 随训练增大是正常现象**（不代表训练出问题）。这是因为 GRPO 的 loss 正比于 KL 散度，随着模型偏离初始策略，loss 自然升高。真正需要关注的指标是 **reward**，它应随训练持续上升。

### 奖励函数设计原则

1. **可验证性**：优先选择答案可客观验证的任务
2. **密集奖励**：避免过于稀疏的奖励（大多数情况得 0）
3. **多维度覆盖**：结合格式 + 正确性 + 其他维度
4. **规避漏洞**：测试模型是否会「走捷径」获得高奖励

---

**下一节**：通过完整的端到端实战练习，在 SmolLM2 模型上运行 GRPO 训练。